# ImmigrationNavigator — RAG Pipeline

**UC Berkeley MIDS Capstone 2026** — Team: Ale, Clover, Duc, Rohan

---

## What this notebook does

Builds the full RAG pipeline over the USCIS corpus from Notebook 1.
Combines the best components from both Ale and Clover's implementations.

## Pipeline architecture

```
S3 corpus (raw_docs.json)
      ↓
Text cleaning + SentenceSplitter chunking
      ↓
Embeddings → ChromaDB vector store
      ↓
Synonym expansion (Clover) + situation-aware query (Ale)
      ↓
Retrieval → topic relevance reranking (Clover)
      ↓
LLM → cited, personalized answer
```

## Model configuration

Set `USE_OPENAI` in Cell 2 to switch between models:

| Setting | Embeddings | LLM | Cost |
|---------|-----------|-----|------|
| `USE_OPENAI = False` | FastEmbed BAAI/bge-small-en-v1.5 | Groq Llama 3.3 70B | Free |
| `USE_OPENAI = True` | OpenAI text-embedding-3-small | GPT-4o-mini | ~$5 total |

Run the benchmark in Cell 9 with both settings and compare results.

## Prerequisites

Run **Notebook 1** first. Copy `synonyms.py` to `/home/sagemaker-user/`.

## 1. Install Dependencies

In [33]:
!pip install langchain langchain-groq langchain-community langchain-openai \
             chromadb fastembed tiktoken boto3 openai llama-index-core \
             llama-index-embeddings-openai llama-index-llms-openai -q

## 2. Imports and Model Configuration

All model choices are controlled by the `USE_OPENAI` flag — change it here and every downstream cell adapts automatically.

API keys are loaded from AWS Secrets Manager — no credentials in code.

In [34]:
import os, re, json, uuid, boto3
from datetime import datetime
from dataclasses import dataclass, field

# ── Model configuration ──────────────────────────────────────
# Set to True to use OpenAI (embeddings + GPT-4o-mini)
# Set to False to use Groq (FastEmbed + Llama 3.1 70B)
USE_OPENAI = True  # ← change this to switch models
# ─────────────────────────────────────────────────────────────

from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
import chromadb
from fastembed import TextEmbedding

# AWS
def get_secret(secret_name):
    client = boto3.client("secretsmanager", region_name="us-east-1")
    response = client.get_secret_value(SecretId=secret_name)
    return json.loads(response["SecretString"])

secrets = get_secret("immigration-navigator/groq")
os.environ["GROQ_API_KEY"] = secrets["GROQ_API_KEY"]

if USE_OPENAI:
    openai_secrets = get_secret("immigration-navigator/openai")
    os.environ["OPENAI_API_KEY"] = openai_secrets["OPENAI_API_KEY"]
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    print("✅ Using OpenAI (text-embedding-3-small + GPT-4o-mini)")
else:
    print("✅ Using Groq (FastEmbed BAAI/bge-small-en-v1.5 + Llama 3.1 70B)")

s3 = boto3.client("s3", region_name="us-east-1")
S3_BUCKET = "immigration-navigator-data"

✅ Using OpenAI (text-embedding-3-small + GPT-4o-mini)


## 3. Load Corpus from S3

Loads the 3,500+ document corpus from S3 and filters to documents with enough content for meaningful RAG (>200 words).

> **Label fix:** Volume 7 Part A in the USCIS site is Adjustment of Status, not OPT. Real OPT content is in `f1_chapter5` (renamed to `opt_and_stem_opt`). This was discovered during EDA and is corrected here.

In [35]:
response = s3.get_object(Bucket=S3_BUCKET, Key="raw_docs.json")
all_docs = json.loads(response["Body"].read().decode("utf-8"))
good_docs = [d for d in all_docs if d["word_count"] > 200]

# Fix mislabeled docs — Volume 7 Part A = Adjustment of Status, not OPT
label_fixes = {
    "opt_overview": "adj_status_overview",
    "opt_chapter1": "adj_status_ch1",
    "opt_chapter2": "adj_status_ch2",
    "opt_chapter3": "adj_status_ch3",
    "opt_chapter4": "adj_status_ch4",
    "opt_chapter5": "adj_status_ch5",
    "f1_chapter5":  "opt_and_stem_opt",  # Real OPT content
}
for d in good_docs:
    if d["label"] in label_fixes:
        d["label"] = label_fixes[d["label"]]

print(f"Docs ready for RAG: {len(good_docs)}")
print(f"Total words: {sum(d['word_count'] for d in good_docs):,}")

Docs ready for RAG: 1351
Total words: 762,403


## 4. Text Cleaning

Removes USCIS navigation menus, CFR reference lines, and boilerplate that got scraped along with the actual policy content.

Without this step, chunks like 'Contents / Updates / INA / 8 CFR / Feedback' end up in the vector store and pollute retrieval results.

In [36]:
def clean_text(text):
    """Remove USCIS nav menus, CFR references, and boilerplate from scraped HTML text."""
    patterns = [
        r'Policy Manual\s*\n.*?Feedback',
        r'USCIS-PM\s*\n.*?Volume \d+.*?\n',
        r'Affected Sections.*?Volume \d+.*?\n',
        r'Skip to main content.*?secure websites\.',
        r'Countdown to America.*?Minutes',
        r'An official website.*?HTTPS',
        r'\d+\s*USCIS-PM\s*-\s*\n',
        r'Contents\s*\nUpdates\s*\nINA\s*\n8 CFR',
        r'8 CFR \d+\.\d+.*?\n',
        r'INA \d+.*?\n',
    ]
    import re
    for pattern in patterns:
        text = re.sub(pattern, '', text, flags=re.DOTALL | re.IGNORECASE)
    lines = [l for l in text.split('\n') if len(l.split()) >= 4]
    text = '\n'.join(lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

## 5. Chunking

Splits documents into chunks for embedding.

Uses **LlamaIndex SentenceSplitter** (512 tokens, 64 overlap) when available — it respects sentence boundaries so chunks don't cut mid-sentence. Falls back to LangChain's RecursiveCharacterTextSplitter if LlamaIndex isn't installed.

Chunks shorter than 50 words are filtered out — they're usually nav fragments that survived the cleaning step.

In [37]:
# Use LlamaIndex SentenceSplitter for section-aware chunking
# Falls back to LangChain RecursiveCharacterTextSplitter if LlamaIndex unavailable
try:
    from llama_index.core.node_parser import SentenceSplitter as LlamaSplitter
    from llama_index.core import Document as LlamaDoc
    USE_LLAMA_SPLITTER = True
    print("Using LlamaIndex SentenceSplitter")
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    USE_LLAMA_SPLITTER = False
    print("Using LangChain RecursiveCharacterTextSplitter (fallback)")

lc_docs = []
for d in good_docs:
    clean = clean_text(d["text"])
    if len(clean.split()) > 100:
        lc_docs.append(Document(
            page_content=clean,
            metadata={"source": d["source"], "label": d["label"], "url": d.get("url", "")}
        ))

if USE_LLAMA_SPLITTER:
    splitter = LlamaSplitter(chunk_size=512, chunk_overlap=64)
    llama_docs = [LlamaDoc(text=d.page_content, metadata=d.metadata) for d in lc_docs]
    nodes = splitter.get_nodes_from_documents(llama_docs)
    chunks = [Document(page_content=n.text, metadata=n.metadata) for n in nodes]
else:
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150, separators=["\n\n", "\n", ". ", " "])
    chunks = splitter.split_documents(lc_docs)

chunks = [c for c in chunks if len(c.page_content.split()) > 50]
print(f"Documents : {len(lc_docs)}")
print(f"Chunks    : {len(chunks)}")
print(f"Avg size  : {sum(len(c.page_content.split()) for c in chunks) // len(chunks)} words")

Using LlamaIndex SentenceSplitter


Documents : 1351
Chunks    : 3066
Avg size  : 262 words


## 6. Embeddings and Vector Store

Embeds all chunks and stores them in ChromaDB.

Two separate ChromaDB collections are created — one per model — so you can run the benchmark with both without rebuilding:
- `immigration_nav_groq` — FastEmbed embeddings
- `immigration_nav_openai` — OpenAI embeddings

The `if collection.count() == 0` guard means re-running this cell never duplicates chunks.

In [38]:
# Initialize embedding model
if USE_OPENAI:
    from langchain_openai import OpenAIEmbeddings
    embed_fn = OpenAIEmbeddings(model="text-embedding-3-small")
    def get_embedding(texts):
        return embed_fn.embed_documents(texts)
    model_name = "openai/text-embedding-3-small"
else:
    from fastembed import TextEmbedding
    embed_model = TextEmbedding("BAAI/bge-small-en-v1.5")
    def get_embedding(texts):
        return [e.tolist() for e in embed_model.embed(texts)]
    model_name = "fastembed/BAAI-bge-small-en-v1.5"

print(f"✅ Embedding model: {model_name}")

# ChromaDB collection — named by model to allow side-by-side comparison
collection_name = f"immigration_nav_{'openai' if USE_OPENAI else 'groq'}"
chroma_client = chromadb.PersistentClient(path="/home/sagemaker-user/chroma_db")
collection = chroma_client.get_or_create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)
print(f"✅ ChromaDB collection: {collection_name} ({collection.count()} chunks already stored)")

if collection.count() == 0:
    BATCH_SIZE = 50
    docs_text = [c.page_content for c in chunks]
    docs_meta = [c.metadata for c in chunks]
    ids = [str(uuid.uuid4()) for _ in chunks]

    print(f"Inserting {len(chunks)} chunks...")
    for i in range(0, len(chunks), BATCH_SIZE):
        embeddings = get_embedding(docs_text[i:i+BATCH_SIZE])
        collection.add(
            documents=docs_text[i:i+BATCH_SIZE],
            embeddings=embeddings,
            metadatas=docs_meta[i:i+BATCH_SIZE],
            ids=ids[i:i+BATCH_SIZE]
        )
        if (i // BATCH_SIZE) % 5 == 0:
            print(f"  {min(i+BATCH_SIZE, len(chunks))}/{len(chunks)} chunks")

print(f"\n✅ Done — {collection.count()} chunks in ChromaDB")

✅ Embedding model: openai/text-embedding-3-small
✅ ChromaDB collection: immigration_nav_openai (3066 chunks already stored)

✅ Done — 3066 chunks in ChromaDB


## 7. LLM Setup

Initializes the LLM and the citation-enforced prompt template.

The prompt instructs the model to:
- Answer **only** from the retrieved context
- Cite every claim with `[Source: label, url]`
- Decline if context is insufficient — directing the user to their ISO or an attorney
- Personalize the answer based on the user's visa status, degree field, and employer type

In [39]:
if USE_OPENAI:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    llm_name = "OpenAI GPT-4o-mini"
else:
    from langchain_groq import ChatGroq
    llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=secrets["GROQ_API_KEY"], temperature=0)
    llm_name = "Groq Llama 3.3 70B"

print(f"✅ LLM: {llm_name}")

PROMPT = ChatPromptTemplate.from_template("""
You are ImmigrationNavigator, an AI assistant helping international students
navigate U.S. visa processes. Answer ONLY using the provided context.
Cite every claim with [Source: label, url].
If context is insufficient, say:
"I don't have enough information. Please consult your ISO or an immigration attorney."

User Profile:
- Visa status: {visa_status}
- Degree field: {degree_field}
- Graduation date: {graduation_date}
- Employer type: {employer_type}

Context:
{context}

Question: {question}

Answer (personalized, cite every claim):
""")

✅ LLM: OpenAI GPT-4o-mini


## 8. RAG Pipeline with Synonym Expansion and Reranking

The `ask()` function is the full pipeline. Four steps happen before the LLM sees anything:

1. **Synonym expansion** (Clover) — 'work permit' becomes 'employment authorization document'
2. **Profile injection** (Ale) — visa status, degree, employer type added to the query
3. **Vector retrieval** — top-k most similar chunks from ChromaDB
4. **Topic reranking** (Clover) — boosts F-1/OPT/H-1B chunks, penalizes J-1/B-1 chunks

This combination addresses the two main failure modes: vocabulary mismatch (user terms vs. legal terms) and topic drift (retrieving irrelevant visa category content).

In [40]:
# Import Clover's synonym expander
# Make sure synonyms.py is in the same directory or on the Python path
try:
    import sys
    sys.path.append('/home/sagemaker-user')
    from synonyms import expand_query
    HAS_SYNONYMS = True
    print("✅ synonyms.py loaded")
except ImportError:
    HAS_SYNONYMS = False
    print("⚠️  synonyms.py not found — running without query expansion")


def ask(question: str, profile: dict, n_results: int = 5) -> str:
    """
    Full RAG pipeline with synonym expansion + situation-aware retrieval.

    Step 1: Expand query using synonym registry (legal term normalization)
    Step 2: Inject user profile into expanded query
    Step 3: Embed and retrieve top-k chunks
    Step 4: Apply relevance reranking (boost F-1/OPT/H-1B, penalize J-1 etc.)
    Step 5: Generate cited, personalized answer

    Args:
        question  (str) : User's natural language question.
        profile   (dict): visa_status, degree_field, graduation_date, employer_type.
        n_results (int) : Chunks to retrieve before reranking.
    """
    # Step 1 — Synonym expansion
    expanded = expand_query(question) if HAS_SYNONYMS else question

    # Step 2 — Inject profile
    enriched_query = (
        f"{expanded} | "
        f"visa: {profile.get('visa_status','')} | "
        f"degree: {profile.get('degree_field','')} | "
        f"employer: {profile.get('employer_type','')}"
    )

    # Step 3 — Embed and retrieve
    embeddings = get_embedding([enriched_query])
    results = collection.query(
        query_embeddings=embeddings,
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    docs_r  = results["documents"][0]
    metas_r = results["metadatas"][0]
    scores  = [1 - d for d in results["distances"][0]]

    # Step 4 — Topic-aware reranking (simplified version of Clover's RelevanceReranker)
    RELEVANT   = ["f-1", "f1", "opt", "stem opt", "h-1b", "h1b", "cap-gap",
                  "ead", "employment authorization", "sevis", "dso", "i-765", "i-983"]
    IRRELEVANT = ["j-1", "j1", "b-1", "b1", "h-2b", "l-1", "o-1", "tn visa",
                  "green card", "adjustment of status", "naturalization"]

    reranked = []
    for doc, meta, score in zip(docs_r, metas_r, scores):
        text_lower = doc.lower()
        relevant_hits   = sum(1 for t in RELEVANT   if t in text_lower)
        irrelevant_hits = sum(1 for t in IRRELEVANT if t in text_lower)
        if relevant_hits > 0:
            score *= min(1.0 + 0.1 * relevant_hits, 1.5)
        if irrelevant_hits > 0:
            score *= max(0.7 ** irrelevant_hits, 0.2)
        reranked.append((score, doc, meta))

    reranked.sort(key=lambda x: x[0], reverse=True)

    # Step 5 — Build context and generate answer
    context_parts = [
        f"[Source: {meta['label']}, {meta.get('url','')}]\n{doc}"
        for _, doc, meta in reranked
    ]
    context = "\n\n".join(context_parts)

    chain = PROMPT | llm
    response = chain.invoke({
        "context":         context,
        "question":        question,
        "visa_status":     profile.get("visa_status",     "F-1"),
        "degree_field":    profile.get("degree_field",    "Not specified"),
        "graduation_date": profile.get("graduation_date", "Not specified"),
        "employer_type":   profile.get("employer_type",   "Not specified"),
    })
    return response.content


print("✅ RAG pipeline ready")

✅ synonyms.py loaded
✅ RAG pipeline ready


## 9. Benchmark

Runs the 3 core MVP questions and collects both automatic and manual metrics.

**Automatic metrics (collected here):**
- Response time (seconds)
- Answer length (words)
- Avg retrieval score (0–1, higher = more relevant chunks)
- Estimated token counts

**Manual metrics (fill in after reading each answer):**
- Correctness 1–5: is the answer factually correct per USCIS?
- Citation accuracy 1–5: does the cited source actually support the claim?
- Hallucination True/False: did the model invent anything not in the context?
- Completeness 1–5: did it answer the full question?

**How to run the benchmark:**
1. Run with `USE_OPENAI = False` → results saved to S3 as `benchmark_results_groq.json`
2. Change `USE_OPENAI = True` in Cell 2, restart kernel, run all cells again
3. Results saved as `benchmark_results_openai.json`
4. Compare both files in Notebook 4 or load them side by side here

In [41]:
import time

# ── Benchmark configuration ──────────────────────────────────
test_profile = {
    "visa_status":     "F-1, currently on OPT",
    "degree_field":    "Computer Science (STEM)",
    "graduation_date": "May 2025",
    "employer_type":   "Full-time employer"
}

questions = [
    "When do I need to apply for OPT and what forms do I need?",
    "Am I eligible for a STEM OPT extension and what are the deadlines?",
    "What happens to my status during the H-1B cap-gap period?",
]

model_label = "OpenAI" if USE_OPENAI else "Groq"
print(f"Model: {model_label}")
print(f"Collection: {collection_name} ({collection.count()} chunks)")
print("=" * 60)

# ── Run benchmark ─────────────────────────────────────────────
results_log = []

for i, q in enumerate(questions):
    print(f"\nQ{i+1}: {q}\n")

    # Measure response time
    start = time.time()
    answer = ask(q, test_profile)
    elapsed = time.time() - start

    # Measure retrieval quality
    expanded = expand_query(q) if HAS_SYNONYMS else q
    enriched = (
        f"{expanded} | "
        f"visa: {test_profile['visa_status']} | "
        f"degree: {test_profile['degree_field']}"
    )
    query_emb = get_embedding([enriched])
    retrieval = collection.query(
        query_embeddings=query_emb,
        n_results=5,
        include=["distances"]
    )
    avg_retrieval_score = 1 - (sum(retrieval["distances"][0]) / len(retrieval["distances"][0]))

    # Estimate token counts
    input_tokens_est  = int(len(enriched.split()) * 1.3)
    output_tokens_est = int(len(answer.split()) * 1.3)

    # Log results
    results_log.append({
        "model":               model_label,
        "question":            q,
        "response_time_sec":   round(elapsed, 2),
        "answer_length_words": len(answer.split()),
        "avg_retrieval_score": round(avg_retrieval_score, 3),
        "input_tokens_est":    input_tokens_est,
        "output_tokens_est":   output_tokens_est,
        "answer":              answer,
        # Fill these in manually after reading the answer:
        "correctness_1_5":     None,
        "citation_accuracy_1_5": None,
        "hallucination":       None,  # True/False
        "completeness_1_5":    None,
    })

    print(answer)
    print(f"\n── Metrics ──")
    print(f"  Response time    : {elapsed:.2f}s")
    print(f"  Answer length    : {len(answer.split())} words")
    print(f"  Retrieval score  : {avg_retrieval_score:.3f}")
    print(f"  Est. input tokens: {input_tokens_est}")
    print(f"  Est. output tokens: {output_tokens_est}")
    print("=" * 60)

# ── Summary table ─────────────────────────────────────────────
import pandas as pd

summary = pd.DataFrame([{
    "Q": f"Q{i+1}",
    "Response time (s)": r["response_time_sec"],
    "Answer length (words)": r["answer_length_words"],
    "Retrieval score": r["avg_retrieval_score"],
    "Est. input tokens": r["input_tokens_est"],
    "Est. output tokens": r["output_tokens_est"],
} for i, r in enumerate(results_log)])

print(f"\n── {model_label} — Automatic Metrics Summary ──")
print(summary.to_string(index=False))

# Save results to S3 for later comparison
s3_key = f"benchmark_results_{model_label.lower()}.json"
s3.put_object(
    Bucket=S3_BUCKET,
    Key=s3_key,
    Body=json.dumps(results_log, indent=2).encode("utf-8")
)
print(f"\nResults saved to s3://{S3_BUCKET}/{s3_key}")
print("Fill in correctness_1_5, citation_accuracy_1_5, hallucination, completeness_1_5 manually.")


Model: OpenAI
Collection: immigration_nav_openai (3066 chunks)

Q1: When do I need to apply for OPT and what forms do I need?

  📝 Query expanded:
     Original: 'When do I need to apply for OPT and what forms do I need?'
     Expanded: 'When do I need to apply for Optional Practical Training and what forms do I need?'


  📝 Query expanded:
     Original: 'When do I need to apply for OPT and what forms do I need?'
     Expanded: 'When do I need to apply for Optional Practical Training and what forms do I need?'
I don't have enough information. Please consult your ISO or an immigration attorney.

── Metrics ──
  Response time    : 2.71s
  Answer length    : 13 words
  Retrieval score  : 0.670
  Est. input tokens: 35
  Est. output tokens: 16

Q2: Am I eligible for a STEM OPT extension and what are the deadlines?

  📝 Query expanded:
     Original: 'Am I eligible for a STEM OPT extension and what are the deadlines?'
     Expanded: 'Am I eligible for a STEM Optional Practical Training extension and what are the deadlines?'


  📝 Query expanded:
     Original: 'Am I eligible for a STEM OPT extension and what are the deadlines?'
     Expanded: 'Am I eligible for a STEM Optional Practical Training extension and what are the deadlines?'
Yes, you are eligible for a STEM OPT extension since you are currently on OPT, have a degree in Computer Science (a STEM field), and your graduation date is May 2025. To apply for the STEM OPT extension, you must file a Form I-765 and obtain a recommendation from your Designated School Official (DSO) [Source: 3. STEM OPT Extension, https://www.uscis.gov/policy-manual/volume-2-part-f-chapter-5#S-C-3].

You can submit your Form I-765 up to 90 days before your current post-completion OPT EAD expires and no more than 60 days after your DSO enters the STEM OPT recommendation into SEVIS [Source: 3. STEM OPT Extension, https://www.uscis.gov/policy-manual/volume-2-part-f-chapter-5#S-C-3]. If you file your application on time, you may continue working until USCIS makes a decision on you

During the H-1B cap-gap period, your F-1 status and employment authorization are automatically extended if you meet certain conditions. Since you are currently on OPT and have not violated the terms of your F-1 status, you will be granted an extension of your F-1 status and employment authorization if your H-1B petition is timely filed and requests an employment start date of October 1 of the following fiscal year [Source: 1. Automatic “Cap-gap” Extension, https://www.uscis.gov/policy-manual/volume-2-part-f-chapter-5#S-D-1].

The cap-gap period starts when your OPT employment authorization expires and continues until October 1, unless your H-1B petition is rejected, denied, revoked, or withdrawn [Source: 1. Automatic “Cap-gap” Extension, https://www.uscis.gov/policy-manual/volume-2-part-f-chapter-5#S-D-1]. During this time, you are allowed to remain in the U.S., but any unemployment during this period will count toward the cumulative maximum allowed unemployment for your OPT [Source: 1

In [32]:
import pandas as pd

# Load both results from S3
groq = json.loads(s3.get_object(Bucket=S3_BUCKET, Key="benchmark_results_groq.json")["Body"].read())
openai = json.loads(s3.get_object(Bucket=S3_BUCKET, Key="benchmark_results_openai.json")["Body"].read())

# Build comparison table
rows = []
for i, (g, o) in enumerate(zip(groq, openai)):
    rows.append({
        "Question": f"Q{i+1}",
        "Groq — Time (s)": g["response_time_sec"],
        "OpenAI — Time (s)": o["response_time_sec"],
        "Groq — Retrieval": g["avg_retrieval_score"],
        "OpenAI — Retrieval": o["avg_retrieval_score"],
        "Groq — Words": g["answer_length_words"],
        "OpenAI — Words": o["answer_length_words"],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print("\n── Answers side by side ──")
for i, (g, o) in enumerate(zip(groq, openai)):
    print(f"\n{'='*60}")
    print(f"Q{i+1}: {g['question']}")
    print(f"\n--- GROQ ---\n{g['answer']}")
    print(f"\n--- OPENAI ---\n{o['answer']}")

Question  Groq — Time (s)  OpenAI — Time (s)  Groq — Retrieval  OpenAI — Retrieval  Groq — Words  OpenAI — Words
      Q1             1.54               0.74             0.780               0.670           227              13
      Q2             1.18               5.67             0.808               0.716           266             152
      Q3             1.34               6.45             0.798               0.665           217             140

── Answers side by side ──

Q1: When do I need to apply for OPT and what forms do I need?

--- GROQ ---
As an F-1 student who has graduated with a Computer Science degree, you are eligible to apply for Optional Practical Training (OPT) [Source: 2. Post-Completion OPT, https://www.uscis.gov/policy-manual/volume-2-part-f-chapter-5#S-C-2]. You can apply for post-completion OPT no earlier than 90 days prior to your program end date and no later than 60 days after your program end-date [Source: 5. OPT Filing, https://www.uscis.gov/policy-manual/v